In [1]:
# !pip install fabric

In [1]:
from fabric import SerialGroup, ThreadingGroup, Group
import threading
from queue import Queue
global_res = Queue(maxsize=0)
def cpu_utils(c):
    print ("running on {}".format(c.host))
    uname = c.run('uname -s', hide=True)
    if 'Linux' in uname.stdout:
        #command = "top -bn1 | grep \"Cpu(s)\" | awk '{print $2 + $4 \"%\"}'"
        command = "docker ps | grep taquangtrung | wc -l"
        res = c.run(command, hide=True).stdout.strip()
        # print (res)
        global_res.put((int(c.host.split('-')[-1]), res))
        return
    print("No idea how to get disk space on {}!".format(uname))
    
all_hosts = [f'worker-{idx:03d}' for idx in range(1,65)]
print (all_hosts[:5])
group = Group(*all_hosts)
# all_results = group.run("top -bn1 | grep \"Cpu(s)\" | awk '{print $2 + $4 \"%\"}'")
for connection in group:
    print (int(connection.host.split('-')[-1]))
    # all_results.append(cpu_utils(connection))
# print (all_results)
#    print("{}: {}".format(cxn, cpu_utils(cxn)))
threads = list()
for index in range(len(group)):

    x = threading.Thread(target=cpu_utils, args=(group[index],))
    threads.append(x)
    x.start()

for index, thread in enumerate(threads):
    thread.join()
print (global_res.qsize())
print (list(global_res.queue))

['worker-001', 'worker-002', 'worker-003', 'worker-004', 'worker-005']
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
running on worker-001
running on worker-002
running on worker-003
running on worker-004
running on worker-005
running on worker-006
running on worker-007
running on worker-008
running on worker-009
running on worker-010
running on worker-011
running on worker-012
running on worker-013
running on worker-014
running on worker-015
running on worker-016
running on worker-017
running on worker-018
running on worker-019
running on worker-020
running on worker-021
running on worker-022
running on worker-023
running on worker-024
running on worker-025
running on worker-026
running on worker-027
running on worker-028
running on worker-029
running on worker-030
running on worker-031
running on worker-032
running on worker-033
running on worker-03

In [2]:
all_results = list(global_res.queue)
all_results.sort(key=lambda x: x[0])
new_res = list(map(lambda x: float(x[1].replace('%','')), all_results))

print (new_res)
# !pip install matplotlib
import matplotlib.pyplot as plt
plt.ylabel('CPU Utilization %')
plt.xlabel('Worker ID')
plt.bar(list(range(1,len(new_res)+1)),new_res)
plt.show()


NameError: name 'global_res' is not defined

In [7]:
global_res = Queue(maxsize=0)   
def ping(c):
    # print ("running on {}".format(c.host))
    uname = c.run('uname -s', hide=True)
    if 'Linux' in uname.stdout:
        command = "ping google.com | head -n 3"
        res = c.run(command, hide=True).stdout.strip()
        # print (res.split("\n")[-1])
        global_res.put((int(c.host.split('-')[-1]), res.split("\n")[-1]))
        return res
    print("No idea how to get disk space on {}!".format(uname))
all_hosts = [f'worker-{idx:03d}' for idx in range(1,65)]
print (all_hosts[:5])
group = Group(*all_hosts)
threads = list()
for index in range(len(group)):

    x = threading.Thread(target=ping, args=(group[index],))
    threads.append(x)
    x.start()

for index, thread in enumerate(threads):
    thread.join()
print (global_res.qsize())
print (list(global_res.queue))
# all_results = group.run("ping google.com | head -n 3")
# print (all_results)
all_results = list(global_res.queue)
all_results.sort(key=lambda x: x[0])
new_res = list(map(lambda x: float(x[1].split("time=")[1].replace(' ms','')), all_results))
print (new_res)


['worker-001', 'worker-002', 'worker-003', 'worker-004', 'worker-005']
64
[(4, '64 bytes from sf-in-f138.1e100.net (74.125.24.138): icmp_seq=2 ttl=50 time=3.37 ms'), (16, '64 bytes from sb-in-f139.1e100.net (74.125.130.139): icmp_seq=2 ttl=99 time=3.03 ms'), (15, '64 bytes from sf-in-f139.1e100.net (74.125.24.139): icmp_seq=2 ttl=100 time=3.51 ms'), (20, '64 bytes from sf-in-f113.1e100.net (74.125.24.113): icmp_seq=2 ttl=99 time=3.04 ms'), (13, '64 bytes from sb-in-f101.1e100.net (74.125.130.101): icmp_seq=2 ttl=100 time=3.22 ms'), (29, '64 bytes from sb-in-f101.1e100.net (74.125.130.101): icmp_seq=2 ttl=100 time=3.43 ms'), (6, '64 bytes from sf-in-f100.1e100.net (74.125.24.100): icmp_seq=2 ttl=99 time=3.22 ms'), (5, '64 bytes from sf-in-f101.1e100.net (74.125.24.101): icmp_seq=2 ttl=100 time=2.84 ms'), (14, '64 bytes from sf-in-f139.1e100.net (74.125.24.139): icmp_seq=2 ttl=100 time=3.12 ms'), (35, '64 bytes from sb-in-f138.1e100.net (74.125.130.138): icmp_seq=2 ttl=99 time=3.03 ms'),

In [4]:
# import os
# solidifi_path = "/users/minh/github/smartbench-dataset/solidity/solidifi++"
# sub_folders = ["Overflow-Underflow", "Re-entrancy", "Timestamp-Dependency", "Unchecked-Send", "Unhandled-Exceptions", "tx.origin"]
# def build_file_list_solidifi(solidifi_path):
#     file_list = []
#     for sub_folder in sub_folders:
#         for filename in os.listdir(os.path.join(solidifi_path, sub_folder)):
#             if filename.endswith(".sol"):
#                 file_list.append(os.path.join(sub_folder, filename))
#     print (len(file_list))
#     print (file_list[:5])
#     print (file_list[-5:])
#     return file_list
# file_list = build_file_list_solidifi(solidifi_path)
# with open("SOLIDIFI.list", "w") as f:
#     for file in file_list:
#         f.write(file+"\n")




266
['Overflow-Underflow/buggy_11.sol', 'Overflow-Underflow/buggy_25.sol', 'Overflow-Underflow/buggy_16.sol', 'Overflow-Underflow/buggy_26.sol', 'Overflow-Underflow/buggy_24.sol']
['tx.origin/buggy_3.sol', 'tx.origin/buggy_45.sol', 'tx.origin/buggy_40.sol', 'tx.origin/buggy_43.sol', 'tx.origin/buggy_29.sol']


In [41]:
tools = ['sfuzz','smartfuzz', 'ilf', 'confuzzius', 'smartian', 'mythril']
import numpy as np
# max_workers = 50
# workers_per_tool = max_workers//len(tools)
# print (tool_id, workers_per_tool )
# worker_list = list(range(tool_id*workers_per_tool, (tool_id+1)*workers_per_tool))
# print (worker_list)
def split_contract_list(contract_list_path, num_workers=10):
    with open(contract_list_path, 'r') as f:
        contract_list = f.readlines()
    total_len = len(contract_list)
    task_list = np.array_split(contract_list, num_workers)
    all_parts_list = []
    for i in range(len(task_list)):
        # print (task_list[i])
        # with open(f'{contract_list_path}_part{i}', 'w') as f:
        #     f.writelines(task_list[i])
        all_parts_list.append(f'{contract_list_path}_part{i}')
    return all_parts_list

all_parts_list = split_contract_list('file_list/B2.list', num_workers=10)

In [46]:
# worker_list = list(range(10,60))#[::5]
tools = ['sfuzz','mythril','smartfuzz', 'ilf', 'confuzzius', 'smartian']
worker_list = list(range(1,61)) #list(range(1,21)) + list(range(31,61)) # no smartfuzz
host_list = [f'worker-{idx:03d}' for idx in worker_list]
num_workers = 10
print (worker_list)
print (host_list)
worker_params = {}
for idx, worker_id in enumerate(worker_list):
    # print (idx//num_workers)
    # print (worker_id, num_workers)
    worker_params[f'worker-{worker_id:03d}'] = { 'tool': tools[(worker_id-1)//num_workers],
                            'file': all_parts_list[(worker_id-1)%num_workers]
                          }
from pprint import pprint
pprint (worker_params)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
['worker-001', 'worker-002', 'worker-003', 'worker-004', 'worker-005', 'worker-006', 'worker-007', 'worker-008', 'worker-009', 'worker-010', 'worker-011', 'worker-012', 'worker-013', 'worker-014', 'worker-015', 'worker-016', 'worker-017', 'worker-018', 'worker-019', 'worker-020', 'worker-021', 'worker-022', 'worker-023', 'worker-024', 'worker-025', 'worker-026', 'worker-027', 'worker-028', 'worker-029', 'worker-030', 'worker-031', 'worker-032', 'worker-033', 'worker-034', 'worker-035', 'worker-036', 'worker-037', 'worker-038', 'worker-039', 'worker-040', 'worker-041', 'worker-042', 'worker-043', 'worker-044', 'worker-045', 'worker-046', 'worker-047', 'worker-048', 'worker-049', 'worker-050', 'worker-051', 'worker-052', 'worker-053', 'worker-054', 'worker-055

In [47]:

from paramiko.client import SSHClient
import paramiko
import threading
MAX_JOBS = 20
TIME_OUT = 60
# benchmark_path = 'benchmarks/solidity/solidifi++'
benchmark_path = 'benchmarks/solidity/smartian/B2/'
part_file_prefix = '/users/minh/github/smartbench-runner/'
extra_args = "--solc-version 0.4.25"
# result_dir = "/users/minh/github/smartbench-results/ase-23/B3_run3_test_1min"
result_dir_1 = "results/test_B2_60_2runs/run1"
result_dir_2 = "results/test_B2_60_2runs/run2"
# result_dir_3 = "results/B3_3runs/B3_test_1min_run3"
tool_timeout = TIME_OUT*4
# new_host_list = host_list
def run_analysis_list(host):
    print ("running on {}".format(host))
    parts_file = worker_params.get(host).get('file')
    tool_name = worker_params.get(host).get('tool')
    smart_bench_run_1 = f"./smartbench.sh analyze -t {tool_name} --benchmark-dir {benchmark_path} --timeout {TIME_OUT} --jobs {MAX_JOBS} " + \
                f"--target-contracts-file {part_file_prefix+parts_file} {extra_args} --result-dir {result_dir_1} --only-create-containers --install-remote-docker "
    smart_bench_run_2 = f"./smartbench.sh analyze -t {tool_name} --benchmark-dir {benchmark_path} --timeout {TIME_OUT} --jobs {MAX_JOBS} " + \
                f"--target-contracts-file {part_file_prefix+parts_file} {extra_args} --result-dir {result_dir_2} --only-create-containers --install-remote-docker "
    # smart_bench_run_3 = f"./smartbench.sh analyze -t {tool_name} --benchmark-dir {benchmark_path} --timeout {TIME_OUT} --jobs {MAX_JOBS} " + \
    #             f"--target-contracts-file {part_file_prefix+parts_file} {extra_args} --result-dir {result_dir_3} --only-create-containers --install-remote-docker"
    command = f"cd /data/minh/smartbench-runner ; {smart_bench_run_1} ; {smart_bench_run_2}" 
    # command = f"source ~/miniconda3/bin/activate py39-smartbench && cd ~/github/smartbench-runner/ && ./smartbench.sh analyze -t {tool_name} " + \
    #             f"-f {benchmark_path} --timeout {TIME_OUT} --jobs {MAX_JOBS} " + \
    #             f"--target-contracts-file {parts_file} {extra_args} --result-dir {result_dir} --create-docker-container"
    print (command)
    # res = c.run(command, disown=True)
    # print (res)
    # return res
    client = SSHClient()
    client.load_system_host_keys()
    client.set_missing_host_key_policy(paramiko.client.AutoAddPolicy)
    client.connect(host)
    stdin, stdout, stderr = client.exec_command(command, get_pty=True)
    for line in stdout.readlines():
        print (host, line)
    for line in stderr.readlines():
        print (host, line)
    return
# group = Group(*new_host_list)
threads = list()
for index in range(len(host_list)):
    # run_analysis_list(group[index], 'smartian')
    x = threading.Thread(target=run_analysis_list, args=(host_list[index], ))
    threads.append(x)
    x.start()
print ("before join")
for index, thread in enumerate(threads):
    thread.join()
print ("done")


running on worker-001
cd /data/minh/smartbench-runner ; ./smartbench.sh analyze -t sfuzz --benchmark-dir benchmarks/solidity/smartian/B2/ --timeout 60 --jobs 20 --target-contracts-file /users/minh/github/smartbench-runner/file_list/B2.list_part0 --solc-version 0.4.25 --result-dir results/test_B2_60_2runs/run1 --only-create-containers --install-remote-docker  ; ./smartbench.sh analyze -t sfuzz --benchmark-dir benchmarks/solidity/smartian/B2/ --timeout 60 --jobs 20 --target-contracts-file /users/minh/github/smartbench-runner/file_list/B2.list_part0 --solc-version 0.4.25 --result-dir results/test_B2_60_2runs/run2 --only-create-containers --install-remote-docker 
running on worker-002
cd /data/minh/smartbench-runner ; ./smartbench.sh analyze -t sfuzz --benchmark-dir benchmarks/solidity/smartian/B2/ --timeout 60 --jobs 20 --target-contracts-file /users/minh/github/smartbench-runner/file_list/B2.list_part1 --solc-version 0.4.25 --result-dir results/test_B2_60_2runs/run1 --only-create-contain

In [1]:
#install docker containers
import paramiko
import json
import threading
from paramiko.client import SSHClient
import paramiko
host_list = [f'worker-{idx:03d}' for idx in range(3,5)]
base_idx = 1
failed_workers = ['worker-038','worker-039'] # no name resolution
worker_ip_mapping = json.load(open('worker_ip.json'))
def install_docker(host, tool, idx):
    print ("running on {}".format(host))
    # parts_file = worker_params.get(host)
    if host in failed_workers:
        source_worker_id = f"worker-{base_idx+idx:03d}"
        g2_username = f"minh@{worker_ip_mapping.get(source_worker_id)}" 
    else:
        g2_username = f"minh@worker-{base_idx+idx:03d}"
    command = f"cd ~/github/smartbench-runner/ && ./scripts/install-tool-docker-scp.sh -t {tool} -n 1 --force-install --use-g2-images --g2-user-name {g2_username}"
    print (command) 
    client = SSHClient()
    client.load_system_host_keys()
    client.set_missing_host_key_policy(paramiko.client.AutoAddPolicy)
    client.connect(host)
    stdin, stdout, stderr = client.exec_command(command,get_pty=True)
    for line in stdout.readlines():
        print (host, line)
    for line in stderr.readlines():
        print (host, line)
# group = Group(*new_host_list)
def deploy_host_list (host_list):
    threads = list()
    for index in range(len(host_list)):
        # run_analysis_list(group[index], 'smartian')
        x = threading.Thread(target=install_docker, args=(host_list[index], 'smartfuzz',index))
        threads.append(x)
        x.start()
    print ("before join")
    for index, thread in enumerate(threads):
        thread.join()
    print ("done")

def deploy_host_range(host_range):
    start_host = host_range[0]
    deployed_host = start_host - base_idx
    print ("start host ", start_host, " deployed host ", deployed_host)
    assert (deployed_host >= len(host_range))
    host_list = [f'worker-{idx:03d}' for idx in host_range]
    deploy_host_list(host_list)
def deploy_all(start,end):
    curr_idx = start
    deployed_host = start - base_idx
    while curr_idx < end:
        end_idx = min (end, curr_idx +  deployed_host)
        deploy_host_range(list(range(curr_idx, end_idx)))
        curr_idx = end_idx
        deployed_host = curr_idx - base_idx

deploy_all(2, 61)


start host  2  deployed host  1
running on worker-002
cd ~/github/smartbench-runner/ && ./scripts/install-tool-docker-scp.sh -t smartfuzz -n 1 --force-install --use-g2-images --g2-user-name minh@worker-001
before join
worker-002 Preapre building Docker containers for: smartfuzz

worker-002 =============================================

worker-002 Pulling Docker image from SBIP G2 for: smartfuzz...

worker-002 

worker-002 rsync minh@worker-001:/data/minh/smartbench-images/docker_image_smartfuzz.tar.gz /data/minh/smartbench-images/docker_image_smartfuzz.tar.gz

worker-002 

026ad140: Loading layer  6.408MB/6.408MB

71bac6b8: Loading layer  13.57MB/13.57MB

6ebd08a5: Loading layer  41.51MB/41.51MB

05ce6b0f: Loading layer  154.3MB/154.3MB

The image taquangtrung/smartfuzz:latest already exists, renaming the old one with ID sha256:0ade6e4080edaac84ac8a628c7cf1378a281b91686fa3ba2d671acbabddf5614 to empty string

worker-002 Loaded image: taquangtrung/smartfuzz:latest

worker-002 

worker-00

In [1]:
# setup github repo & data
import json
import threading
from paramiko.client import SSHClient
import paramiko
base_idx = 1
failed_workers = ['worker-038','worker-039'] # no name resolution
worker_ip_mapping = json.load(open('worker_ip.json'))
def install_smartbench(host, tool, idx):
    print ("running on {}".format(host))
    # parts_file = worker_params.get(host)
    if host in failed_workers:
        source_worker_id = f"worker-{base_idx+idx:03d}"
        g2_username = f"minh@{worker_ip_mapping.get(source_worker_id)}" 
    else:
        g2_username = f"minh@worker-{base_idx+idx:03d}"    
    command = f"scp {g2_username}:/data/minh/smartbench-runner.tar.gz /data/minh/smartbench-runner.tar.gz ; cd /data/minh ; tar -xf smartbench-runner.tar.gz"
    print (f"{host}:{command}") 
    client = SSHClient()
    client.load_system_host_keys()
    client.set_missing_host_key_policy(paramiko.client.AutoAddPolicy)
    client.connect(host)
    stdin, stdout, stderr = client.exec_command(command,get_pty=True)
    for line in stdout.readlines():
        print (host, line)
    for line in stderr.readlines():
        print (host, line)
    return
def deploy_host_list (host_list):
    threads = list()
    for index in range(len(host_list)):
        # run_analysis_list(group[index], 'smartian')
        x = threading.Thread(target=install_smartbench, args=(host_list[index], 'smartfuzz',index))
        threads.append(x)
        x.start()
    print ("before join")
    for index, thread in enumerate(threads):
        thread.join()
    print ("done")

def deploy_host_range(host_range):
    start_host = host_range[0]
    deployed_host = start_host - base_idx
    print ("start host ", start_host, " deployed host ", deployed_host)
    assert (deployed_host >= len(host_range))
    host_list = [f'worker-{idx:03d}' for idx in host_range]
    deploy_host_list(host_list)
def deploy_all(start,end):
    curr_idx = start
    deployed_host = start - base_idx
    while curr_idx < end:
        end_idx = min (end, curr_idx +  deployed_host)
        deploy_host_range(list(range(curr_idx, end_idx)))
        curr_idx = end_idx
        deployed_host = curr_idx - base_idx

deploy_all(2, 4)

start host  2  deployed host  1
running on worker-002
worker-002:scp minh@worker-001:/data/minh/smartbench-runner.tar.gz /data/minh/smartbench-runner.tar.gz ; cd /data/minh ; tar -xf smartbench-runner.tar.gz
before join
smartbench-runner.tar.gz                      100% 1062MB 416.6MB/s   00:02    

done
start host  3  deployed host  2
running on worker-003
worker-003:scp minh@worker-001:/data/minh/smartbench-runner.tar.gz /data/minh/smartbench-runner.tar.gz ; cd /data/minh ; tar -xf smartbench-runner.tar.gz
before join
smartbench-runner.tar.gz                      100% 1062MB 394.1MB/s   00:02    

done


In [ ]:
import